In [2]:
import os

REPO_DIR = "/content/ANN2SNN-Conversion"
REPO_URL = "https://github.com/lazylettuce1/ANN2SNN-Conversion.git" # Update with your exact GitHub URL

if not os.path.exists(REPO_DIR):
    # Clone directly into REPO_DIR
    !git clone {REPO_URL} {REPO_DIR}
else:
    # Pull latest changes if it already exists
    %cd {REPO_DIR}
    !git pull

# Verify existence before changing directory
if os.path.exists(REPO_DIR):
    os.chdir(REPO_DIR)
    print(f"Current working directory: {os.getcwd()}")
else:
    print(f"Error: Directory {REPO_DIR} still does not exist. Check your repository URL.")

fatal: could not create leading directories of '/content/ANN2SNN-Conversion': Permission denied


Error: Directory /content/ANN2SNN-Conversion still does not exist. Check your repository URL.


In [27]:
# Create a specific folder if it doesn't exist, and move into it
import os
project_path = '/content/drive/MyDrive/ColabProjects/CS6886_A4'
os.makedirs(project_path, exist_ok=True)

%cd {project_path}
%ls

In [28]:
!python3 setup.py install

In [29]:
import torch
import torchvision
import torch.nn as nn
import spikingjelly
from spikingjelly.activation_based import ann2snn
from tqdm import tqdm
from spikingjelly.activation_based.ann2snn.examples import cnn_mnist as snn 
from spikingjelly.activation_based.ann2snn.sample_models import mnist_cnn
import numpy as np
import matplotlib.pyplot as plt

In [30]:
!nvidia-smi

### **Task 1** is installing all prerequisites correctly so that the following imports run smoothly. (10 marks)

Install the correct version of torch if you wish to use CUDA.

Install the spikingjelly using the setup.py file only and not with pip using the command 'python setup.py install'.

In [31]:
device_used = 'cuda' if torch.cuda.is_available() else 'cpu'
# set Hyperparameters
snn.hyperparameters.T = 20
snn.hyperparameters.batch_size = 100

device =  device_used # use 'cpu' if CUDA not available
download_dataset = True # downloads MNIST
dataset_dir = './spikingjelly/datasets/mnist'
download_model = False # downloads a 3 layer CNN classifier

if download_model:
        print('Downloading SJ-mnist-cnn_model-sample.pth...')
        ann2snn.download_url("https://ndownloader.figshare.com/files/34960191", './SJ-mnist-cnn_model-sample.pth')

model = mnist_cnn.CNN().to(device)
model.load_state_dict(torch.load('SJ-mnist-cnn_model-sample.pth', map_location=device))
print(model)

We see the specification of our CNN, having three Conv blocks with [Conv, BatchNorm2d, ReLU, AvgPool2d] modules.

### **Task 2** is calculating the number of MAC operations in each Conv layer. (10 Marks)


Below we take the first two Conv blocks (first 8 modules) as a backbone and the rest of the model as the head.

We freeze the backbone, convert the head to an SNN and evaluate using snn.main.

In [32]:
# TASK 2: number of mac operations in each convolutional layer
# Starting input size for MNIST
in_H, in_W = 28, 28
for i, layer in enumerate(model.network):
    import math
    if isinstance(layer, nn.Conv2d):
        C_in  = layer.in_channels
        C_out = layer.out_channels
        kH, kW = layer.kernel_size   # kernel_size is always stored as a tuple
        sH, sW = layer.stride
        pH, pW = layer.padding

        # out size, since nn.Conv2d doesn't store in or out sizes
        out_H = math.floor((in_H - kH + 2 * pH) / sH) + 1
        out_W = math.floor((in_W - kW + 2 * pW) / sW) + 1

        macs = C_in * kH * kW * out_H * out_W * C_out

        print(f"Layer {i}: Conv2d(Cin={C_in}, Cout={C_out}, k={kH}x{kW}, s={sH}x{sW}, out={out_H}x{out_W})")
        print(f"  MACs: {macs:,}")
        in_H, in_W = out_H, out_W

    elif isinstance(layer, nn.AvgPool2d):
        # AvgPool changes size
        kH = layer.kernel_size
        sH = layer.stride
        pH = layer.padding

        in_H = math.floor((in_H - kH + 2 * pH) / sH) + 1
        in_W = math.floor((in_W - kH + 2 * pH) / sH) + 1
        # (assumes square kernel, which it is here: 2x2)

In [34]:
split_index = 8
backbone = model.network[:split_index]
head = model.network[split_index:]
snn_head, vals = snn.main(eval_fn = snn.val, 
    backbone = backbone, 
    head = head, 
    conversion = snn.conversion_job, 
    download_dataset = False,
    device = device_used,
    dataset_dir = dataset_dir)
print(vals)

main() has returned the converted model head and also its accuracy for number of timesteps from 1 to T.

1. We provided a conversion algorithm to snn.main. Go to the file spikingjelly\activation_based\ann2snn\examples\cnn_mnist.py to see how this is defined in snn.conversion_job().
2. You should see some commented out code snippets for other possible conversion jobs. 
3. You can make your own conversion job using any one of those and pass it into snn.main()


### **Task 3** is plotting accuracy vs time steps for all the mentioned conversion jobs in one plot. (10 Marks)
Pick the conversion job that gives best results for you. Stick to this algorithm for all further tasks.

In [36]:
# Task 3: Compare different conversion modes
split_index = 8
backbone = model.network[:split_index]
head = model.network[split_index:]

mode_names = ['max', '99.9%', '1/2 max', '1/4 max']
results = {}

for mode_sel in [0, 1, 2, 3]:
    print(f"\n{'='*60}")
    print(f"Running conversion with mode_sel={mode_sel} ({mode_names[mode_sel]})")
    print('='*60)
    
    snn_head, vals = snn.main(
        eval_fn=snn.val, 
        backbone=backbone, 
        head=head,
        mode_sel=mode_sel,
        conversion=snn.conversion_job, 
        download_dataset=(mode_sel == 0),  # only download on first iteration
        device=device_used,
        dataset_dir=dataset_dir
    )
    results[mode_names[mode_sel]] = vals
    print(f"Final accuracy at T={snn.hyperparameters.T}: {vals[-1]:.4f}")

# Plot accuracy vs timesteps for all modes
plt.figure(figsize=(10, 6))
timesteps = range(1, snn.hyperparameters.T + 1)

for mode_name, vals in results.items():
    plt.plot(timesteps, vals, marker='o', label=mode_name, linewidth=2, markersize=4)

plt.xlabel('Timesteps (T)', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Accuracy vs Timesteps for Different Conversion Modes', fontsize=14)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Report best mode
best_mode = max(results.items(), key=lambda x: x[1][-1])
print(f"\nBest conversion mode: {best_mode[0]} with accuracy {best_mode[1][-1]:.4f} at T={snn.hyperparameters.T}")


### **Task 4** is plotting accuracy vs time steps for the original CNN, 1 conv block converted, 2 blocks converted, 3 blocks in one plot. (10 Marks)
Now let us look at the converted model.

In [ ]:
print(snn_head)

SpikingJelly has changed the forward pass of the model significantly. 

In the framework the original Conv2d layer is used but only to simulate the scaling for the inputs to the IF neurons striped across time.
Hence we can use the number of spikes generated multiplied by the fan-out of the layer to estimate the number of accumulate operations (refer https://doi.org/10.3389/fnins.2017.00682)

We use val() in main() after conversion is complete. It evaluates the model on the train dataset and also registers a hook that stores spike counts of a IFNode layer in the model into snn.spike_counts. Dividing this by the size of the dataset provides an average for spike count per layer in a forward pass.

In [ ]:
print(snn.spike_counts)


Go to the file spikingjelly\activation_based\ann2snn\examples\cnn_mnist.py to see how snn.val() is defined

We use val() in main() after conversion is complete. It evaluates the model on the train dataset and also registers a hook that stores spike counts of a IFNode layer in the model into snn.spike_counts.

Write your own custom_val() functions so that you can register hooks to the IFNode() modules for storing spike counts into a dictionary/list for different converted models. (You should need to make minimal changes to val(). Printing a model out might help you figure out how to assign the right hooks.)

### Task 5 is getting average spike count in all layers for a forward pass of a fully converted SNN. (10 Marks)

For a fully converted SNN, use snn.main(custom_val, backbone = None, head = model.network).

**Assume that energy cost of a MAC is 4.6nJ and cost of an accumulate op is 0.6nJ.** Using these values you can estimate the energy savings in converting a layer.

### Task 6 is reporting the best accuracy for an energy savings of 90% or higher. (15 Marks)

### Task 7 is reporting the best energy savings you could achieve from ANN conversion for a loss in accuracy less than 5% from the original CNN. (15 Marks)


Remember, you can scale number of time steps (snn.hyperparameters.T) to scale both computation and accuracy.




So far we have used a really optimistic cost model which only considers energy required for compute operations; we have ignored the costs of writing our compute results to memory.
In reality activation maps are typically too large to fit entirely in CPU cache, especially in vision models. DRAM writes of a 64 byte vector are ~10-50nj and could easily be upwards of 100nj when activating different DRAM rows.

Efficient vision model pipelines try to take maximum advantage of tiling and operator fusion so that computation is focused on smaller regions of the activation map which can fit into cache memory. SRAM writes average around ~1-3nj per vector write across different levels, but this approach introduces a tradeoff between memory traffic and recomputation.

Assume you have a compute pipeline for vision models with high bandwidth memory where performing writes costs 25nj per 64 bytes on average across the available memory hierarchy. Ignore the costs of loading weights, hence only your activation maps contribute to memory-related energy costs. Treat the Conv2d, BatchNorm, ReLU and AvgPool operations within a block as fused for the purpose of memory accounting.

### Task 8 is calculating the memory-related energy costs of each layer in one forward pass of the CNN model. (5 Marks)

Assume these are also the memory-related energy cost of one timestep of running a converted SNN layer. Running the model for multiple time steps incurs this cost at each timestep.

### Task 9 is to report the best energy savings you could achieve from ANN conversion for a loss in accuracy less than 5% with the memory-aware cost model. (15 Marks)

This should highlight the importance of neuromorphic memory innovations like in-memory compute and asynchronous core design.
